# MVP Pipeline: Adaptive Augmentations for Small Objects

This notebook is the main entrypoint for testing MVP in Google Colab.

Pipeline steps:
1. Install dependencies.
2. Validate dataset structure.
3. Analyze dataset and save stats.
4. Generate adaptive augmentation policy.
5. (Optional) Run training modes.
6. (Optional) Convert to COCO and run COCOeval.

In [1]:
# Install dependencies in Colab runtime.
%pip install -q ultralytics albumentations opencv-python pycocotools pyyaml numpy pandas matplotlib tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 19.6 MB/s eta 0:00:00a 0:00:01


In [3]:
from pathlib import Path
import sys

# 1) Попробуем найти проект в типовых местах
candidates = [
    Path("/content/small_objects_auto_aug"),
    Path("/content/drive/MyDrive/small_objects_auto_aug"),
    Path.cwd(),
    Path.cwd().parent,
]

PROJECT_ROOT = None
for p in candidates:
    if (p / "src").exists() and (p / "configs").exists():
        PROJECT_ROOT = p
        break

# 2) Если не нашли — клонируем в /content (подставь свой URL)
if PROJECT_ROOT is None:
    import subprocess
    repo_url = "https://github.com/<your-user>/<your-repo>.git"  # <-- замени
    subprocess.run(["git", "clone", repo_url, "/content/small_objects_auto_aug"], check=True)
    PROJECT_ROOT = Path("/content/small_objects_auto_aug")

sys.path.insert(0, str(PROJECT_ROOT))
print("Using project root:", PROJECT_ROOT)


CalledProcessError: Command '['git', 'clone', 'https://github.com/<your-user>/<your-repo>.git', '/content/small_objects_auto_aug']' returned non-zero exit status 128.

In [ ]:
# Imports and configuration.
from src.utils.io import load_yaml
from src.data.visdrone_manager import validate_visdrone_yolo_structure, save_validation_report
from src.analysis.dataset_analyzer import DatasetAnalyzerConfig, analyze_dataset
from src.policy.rule_engine import RuleEngineConfig, generate_policy_from_stats, save_policy_artifacts

project_cfg = load_yaml(PROJECT_ROOT / 'configs' / 'project_config.yaml')
dataset_root = Path(project_cfg['dataset']['root'])
if not dataset_root.is_absolute():
    dataset_root = PROJECT_ROOT / dataset_root

splits = tuple(project_cfg['dataset'].get('splits', ['train', 'val']))
image_extensions = tuple(project_cfg['dataset'].get('image_extensions', ['.jpg', '.jpeg', '.png', '.bmp']))

stats_dir = PROJECT_ROOT / 'artifacts' / 'stats'
policy_dir = PROJECT_ROOT / 'artifacts' / 'policy'
stats_dir.mkdir(parents=True, exist_ok=True)
policy_dir.mkdir(parents=True, exist_ok=True)

print('Dataset root:', dataset_root)

In [ ]:
# Step 1-2: validate and analyze dataset.
validation_report = validate_visdrone_yolo_structure(
    dataset_root=dataset_root,
    splits=splits,
    image_extensions=image_extensions,
)
save_validation_report(validation_report, stats_dir / 'validation_report.json')
print('Validation is_valid =', validation_report['is_valid'])

analyzer_cfg = DatasetAnalyzerConfig(
    small_max_area=float(project_cfg['analysis']['small_max_area']),
    medium_max_area=float(project_cfg['analysis']['medium_max_area']),
    tiny_max_area=float(project_cfg['analysis']['tiny_max_area']),
    image_extensions=image_extensions,
    generate_plots=bool(project_cfg['analysis'].get('generate_plots', True)),
)
stats = analyze_dataset(
    dataset_root=dataset_root,
    output_dir=stats_dir,
    splits=splits,
    config=analyzer_cfg,
)
print('Stats saved to:', stats_dir)

In [ ]:
# Step 3: generate adaptive policy and explainability report.
rule_cfg = RuleEngineConfig.from_project_config(project_cfg)
policy, decision_report = generate_policy_from_stats(stats, cfg=rule_cfg)
paths = save_policy_artifacts(policy=policy, decision_report=decision_report, output_dir=policy_dir)

print('Policy JSON:', paths['policy_json'])
print('Policy YAML:', paths['policy_yaml'])
print('Decision report:', paths['decision_report'])
print('Fired rules:', len(decision_report.get('fired_rules', [])))

In [ ]:
# Step 4 (optional): run training suite.
# Switch RUN_TRAINING to True when your data.yaml and GPU setup are ready.
RUN_TRAINING = False

if RUN_TRAINING:
    from src.training.train_runner import TrainRunConfig, run_mvp_training_suite

    train_cfg = TrainRunConfig(
        data_yaml=str(project_cfg['training']['data_yaml']),
        model=str(project_cfg['training']['model']),
        epochs=int(project_cfg['training']['epochs_final']),
        imgsz=int(project_cfg['training']['imgsz']),
        batch=int(project_cfg['training']['batch']),
        device=project_cfg['training'].get('device'),
        workers=int(project_cfg['training']['workers']),
        fraction=1.0,
        project_dir=str(PROJECT_ROOT / 'runs'),
        seed=int(project_cfg['project']['seed']),
        deterministic=bool(project_cfg['project']['deterministic']),
        rect=bool(project_cfg['training'].get('rect', False)),
        multi_scale=bool(project_cfg['training'].get('multi_scale', False)),
        baseline_disable_albumentations=bool(project_cfg['training'].get('baseline_disable_albumentations', True)),
    )

    run_dirs = run_mvp_training_suite(
        config=train_cfg,
        baseline_yaml_path=PROJECT_ROOT / 'configs' / 'baseline.yaml',
        manual_yaml_path=PROJECT_ROOT / 'configs' / 'manual.yaml',
        adaptive_policy_json_path=policy_dir / 'policy_adaptive.json',
        run_ablation=True,
    )
    print(run_dirs)
else:
    print('Training is skipped. Set RUN_TRAINING=True to enable.')

In [ ]:
# Step 5 (optional): COCO conversion and COCOeval.
# Provide directory with YOLO prediction labels (*.txt) for val split.
RUN_EVAL = False
PRED_LABELS_DIR = PROJECT_ROOT / 'runs' / 'detect' / 'predict' / 'labels'  # edit if needed

if RUN_EVAL:
    from src.evaluation.coco_converter import convert_yolo_gt_to_coco, convert_yolo_pred_txt_to_coco
    from src.evaluation.coco_eval_runner import run_coco_eval
    from src.evaluation.metrics_report import build_markdown_report

    eval_dir = PROJECT_ROOT / 'artifacts' / 'eval'
    eval_dir.mkdir(parents=True, exist_ok=True)

    coco_gt_path = eval_dir / 'coco_gt.json'
    coco_dt_path = eval_dir / 'coco_dt.json'

    convert_yolo_gt_to_coco(
        images_dir=dataset_root / 'images' / 'val',
        labels_dir=dataset_root / 'labels' / 'val',
        class_names=project_cfg['dataset']['visdrone_classes'],
        output_path=coco_gt_path,
        image_extensions=image_extensions,
    )
    convert_yolo_pred_txt_to_coco(
        pred_labels_dir=PRED_LABELS_DIR,
        images_dir=dataset_root / 'images' / 'val',
        output_path=coco_dt_path,
        image_extensions=image_extensions,
    )

    metrics = run_coco_eval(
        coco_gt_path=coco_gt_path,
        coco_dt_path=coco_dt_path,
        output_path=eval_dir / 'coco_eval.json',
        use_tiny_eval=True,
        tiny_max_area=float(project_cfg['analysis']['tiny_max_area']),
    )

    report_path = build_markdown_report({'adaptive': metrics}, PROJECT_ROOT / 'artifacts' / 'reports' / 'mvp_report.md')
    print(metrics)
    print('Report:', report_path)
else:
    print('Evaluation is skipped. Set RUN_EVAL=True to enable.')